# Hierarchical Quaternion LSTM for Bitcoin Price Prediction

This notebook runs experiments comparing **6 hierarchical QLSTM variants** on **Hourly BTC data** from LunarCrush.

---

## What is the Hierarchical QLSTM?

The standard QLSTM encodes **4 OHLC features** as a single quaternion:
```
q = Open + High·i + Low·j + Close·k
```

The **hierarchical** model extends this to **16 features** drawn from LunarCrush — a dataset combining price, market, and social sentiment signals. Rather than treating 16 features as one flat vector, we exploit the fact that they naturally cluster into **4 semantic groups of 4**, each encoded as its own quaternion:

| Group | Quaternion | Features |
|-------|-----------|----------|
| **Price** | q₁ = open + high·i + low·j + close·k | Traditional OHLC candle |
| **Market** | q₂ = volume + mcap·i + dominance·j + supply·k | On-chain market fundamentals |
| **Social Activity** | q₃ = active_contrib + new_contrib·i + active_posts·j + new_posts·k | Community engagement volume |
| **Social Sentiment** | q₄ = sentiment + galaxy·i + social_dom·j + interactions·k | Sentiment quality signals |

Each group is processed by its **own independent QLSTM**, then a **fusion layer** combines the 4 group-level representations.

---

## Three Fusion Strategies (Ablation Study)

We test three fusion approaches, each testing a different hypothesis:

1. **Concat → Linear** — Concatenate all 4 group hidden states and project down. Simplest baseline; tests whether hierarchical grouping alone helps.

2. **Group Attention** — Learned attention weights over the 4 groups. The model decides which semantic domain matters most for prediction. *Interpretable*: you can inspect which group the model relies on.

3. **Meta-Quaternion** — Treat the 4 group representations as components of a *new* quaternion (Price=r, Market=i, Social=j, Sentiment=k) and fuse via Hamilton product. *Novel*: inter-group relationships are modeled algebraically.

Each fusion is tested **with and without per-group temporal attention** → **6 variants total**.

---

## Requirements
- GPU Runtime (Runtime → Change runtime type → T4 GPU)
- Google Drive for saving results
- LunarCrush data cache (`lunarcrush_btc_hour_full.csv`) must be accessible

## 1. Environment Setup

In [ ]:
# Clone the thesis repository
!git clone https://github.com/BerkayClik/thesis.git

In [ ]:
%cd /content/thesis

In [ ]:
# Pull latest changes (in case the repo was already cloned)
!git pull

In [ ]:
# Install dependencies
!pip install -q yfinance scipy seaborn

In [ ]:
# Verify GPU is available — hierarchical model has 4 QLSTMs so GPU matters
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Mount Google Drive for persistent result storage
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

GDRIVE_OUTPUT_DIR = "/content/drive/MyDrive/thesis_results_btc_hourly_hier"
os.makedirs(GDRIVE_OUTPUT_DIR, exist_ok=True)

# CUBLAS_WORKSPACE_CONFIG is required for deterministic CUDA operations
os.environ["PYTHONPATH"] = "/content/thesis"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

print(f"Results will be saved to: {GDRIVE_OUTPUT_DIR}")

## 2. Verify the Hierarchical Model

Before running the full experiment, let's do a quick sanity check to make sure all 6 variants instantiate, forward-pass, and back-propagate correctly.

In [ ]:
import sys
sys.path.insert(0, '/content/thesis')

from src.models.hierarchical_qlstm import HierarchicalQLSTM

# Simulate a mini-batch: 4 samples, 72 timesteps (3 days of hourly data), 16 features
batch, seq, features = 4, 72, 16

variants = [
    ('concat',          False, 'Concat fusion, no temporal attention'),
    ('concat',          True,  'Concat fusion + per-group temporal attention'),
    ('group_attention',  False, 'Group attention fusion, no temporal attention'),
    ('group_attention',  True,  'Group attention + per-group temporal attention'),
    ('meta_quaternion',  False, 'Meta-quaternion fusion, no temporal attention'),
    ('meta_quaternion',  True,  'Meta-quaternion + per-group temporal attention'),
]

print(f"{'Variant':<55} {'Output':>12} {'Params':>10}")
print('-' * 80)

for fusion, use_attn, desc in variants:
    model = HierarchicalQLSTM(
        hidden_size=32, num_layers=2, dropout=0.1,
        num_features=16, target_col=3,
        fusion_type=fusion, use_temporal_attention=use_attn,
    )
    x = torch.randn(batch, seq, features)
    out = model(x)

    # Verify backward pass
    loss = ((out - torch.randn(batch, 1)) ** 2).mean()
    loss.backward()
    grad_count = sum(1 for p in model.parameters() if p.grad is not None)
    total_count = sum(1 for _ in model.parameters())
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    assert grad_count == total_count, f"Not all params got gradients: {grad_count}/{total_count}"
    print(f"{desc:<55} {str(out.shape):>12} {params:>10,}")

print('\nAll 6 variants passed forward + backward checks.')

## 3. Inspect the Feature Grouping

Let's verify which LunarCrush features map to which quaternion group. The `feature_cols` in the config selects 16 of the 18 available columns and orders them so that every consecutive block of 4 forms one semantic quaternion.

In [ ]:
from src.data.lunarcrush_api import (
    LUNARCRUSH_ALL_COLUMNS,
    HIERARCHICAL_FEATURE_GROUPS,
    HIERARCHICAL_FEATURE_COLS,
)

print('All 18 LunarCrush columns:')
for i, col in enumerate(LUNARCRUSH_ALL_COLUMNS):
    marker = ' ←' if i in HIERARCHICAL_FEATURE_COLS else ' ✗ (dropped)'
    print(f'  [{i:2d}] {col:<25}{marker}')

print(f'\n16 selected features (ordered by group):')
print(f'  Indices: {HIERARCHICAL_FEATURE_COLS}')

print(f'\nSemantic groups:')
for name, indices in HIERARCHICAL_FEATURE_GROUPS.items():
    cols = [LUNARCRUSH_ALL_COLUMNS[i] for i in indices]
    print(f'  {name:<12} → indices {indices} → {cols}')

## 4. Run Hierarchical QLSTM Experiments

This runs all **6 variants** (3 fusion strategies × 2 attention modes) on hourly BTC data with the 16-feature hierarchical grouping.

**Config files used:**
- Data: `configs/data/hourly/btc_hier.yaml` — loads full LunarCrush data, selects 16 features ordered by semantic group, 72-bar window
- Experiment: `configs/experiments/hourly_hierarchical.yaml` — defines the 6 model variants, all with hidden_size=32, 2 QLSTM layers

**Expected runtime:** ~30–60 min on T4 GPU (4 independent QLSTMs per variant × 6 variants)

In [ ]:
# Run the full hierarchical experiment suite
!python experiments/run_experiments.py \
    --base-config configs/data/hourly/btc_hier.yaml \
    --experiment-config configs/experiments/hourly_hierarchical.yaml

## 5. Generate Visualizations

Uses the same visualization script as the OHLC experiments. Generates:
- Training curves (loss over epochs per variant)
- Metric comparison bar charts (MAPE, Directional Accuracy, Sharpe Ratio)
- Box plots across seeds
- Significance heatmap (p-values vs baseline)
- Parameter efficiency scatter
- Radar chart
- Prediction vs actual price plots

In [ ]:
import glob

# Find the latest hierarchical results file
results_files = glob.glob("/content/thesis/experiments/results/hourly_hierarchical*.json")
results_files = [f for f in results_files if 'intermediate' not in f]

if results_files:
    latest_results = max(results_files, key=os.path.getmtime)
    print(f"Using results file: {latest_results}")

    !python experiments/visualize_results.py \
        --results "{latest_results}" \
        --output experiments/figures_hier
else:
    print("No results file found. Run experiments first (cell above).")

## 6. Copy Results to Google Drive

In [ ]:
import shutil
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_output_dir = f"{GDRIVE_OUTPUT_DIR}/{timestamp}"
os.makedirs(run_output_dir, exist_ok=True)

# Copy JSON results
results_src = "/content/thesis/experiments/results/"
if os.path.exists(results_src):
    shutil.copytree(results_src, f"{run_output_dir}/results", dirs_exist_ok=True)
    print(f"Results copied to: {run_output_dir}/results")

# Copy figures
figures_src = "/content/thesis/experiments/figures_hier"
if os.path.exists(figures_src):
    shutil.copytree(figures_src, f"{run_output_dir}/figures", dirs_exist_ok=True)
    print(f"Figures copied to: {run_output_dir}/figures")

print(f"\n{'=' * 50}")
print(f"All files saved to: {run_output_dir}")
print('=' * 50)

In [ ]:
# List saved files
print("\nSaved Results:")
!ls -la {run_output_dir}/results/ 2>/dev/null || echo "No results folder"

print("\nSaved Figures:")
!ls -la {run_output_dir}/figures/ 2>/dev/null || echo "No figures folder"

## 7. Display Key Figures

In [ ]:
from IPython.display import Image, display

figures_dir = "/content/thesis/experiments/figures_hier"

key_figures = [
    "metric_comparison.png",
    "box_plots.png",
    "radar_chart.png",
    "parameter_efficiency.png",
    "predictions_all_models.png",
    "training_curves.png",
]

for fig in key_figures:
    fig_path = f"{figures_dir}/{fig}"
    if os.path.exists(fig_path):
        print(f"\n{'=' * 50}")
        print(f"{fig}")
        print(f"{'=' * 50}")
        display(Image(filename=fig_path, width=800))

## 8. Group Attention Weight Analysis

For the **group attention** variants, we can inspect which semantic group the model learned to rely on most heavily. This is one of the key interpretability advantages of the hierarchical architecture.

The attention weights sum to 1.0 across the 4 groups for each sample. A higher weight means the model considers that group more predictive of the next close price.

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import sys
sys.path.insert(0, '/content/thesis')

from src.data.loader import load_sp500_data, dataframe_to_tensor
from src.data.preprocessing import select_features, preprocess_data_ratio
from src.data.dataset import SP500Dataset
from src.models.hierarchical_qlstm import HierarchicalQLSTM
from torch.utils.data import DataLoader

# ---- Load data using the same config as the experiment ----
import yaml
with open('configs/data/hourly/btc_hier.yaml') as f:
    cfg = yaml.safe_load(f)

dc = cfg['data']
df = load_sp500_data(
    data_path=dc.get('data_path'), source=dc['source'],
    coin=dc.get('coin', 'btc'), cache_dir=dc.get('cache_dir'),
)
df = df.ffill().bfill()
data, dates = dataframe_to_tensor(df)
data = select_features(data, dc['feature_cols'])

processed = preprocess_data_ratio(
    data, dates,
    train_ratio=dc['train_ratio'],
    val_ratio=dc['val_ratio'],
    test_ratio=dc['test_ratio'],
    target_col=dc['target_col'],
)
test_ds = SP500Dataset(processed['test_data'], window_size=dc['window_size'], target_col=dc['target_col'])
test_loader = DataLoader(test_ds, batch_size=64)

# ---- Build a group-attention model and collect weights ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = HierarchicalQLSTM(
    hidden_size=32, num_layers=2, dropout=0.1,
    num_features=16, target_col=dc['target_col'],
    fusion_type='group_attention', use_temporal_attention=False,
).to(device)

# Load best checkpoint if it exists
ckpt_path = 'experiments/results/checkpoints/hier_qlstm_group_attn/seed_42/best_model.pt'
if os.path.exists(ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location=device)['model_state_dict'])
    print(f'Loaded checkpoint: {ckpt_path}')
else:
    print(f'No checkpoint found at {ckpt_path} — using random weights (run experiments first).')

model.eval()
all_weights = []

with torch.no_grad():
    for x, _ in test_loader:
        x = x.to(device)
        _, weights = model(x, return_group_weights=True)
        all_weights.append(weights.cpu())

all_weights = torch.cat(all_weights)  # (N, 4)
group_names = ['Price', 'Market', 'Social', 'Sentiment']
mean_weights = all_weights.mean(dim=0).numpy()
std_weights = all_weights.std(dim=0).numpy()

print(f"\n{'Group':<14} {'Mean Weight':>12} {'Std':>10}")
print('-' * 38)
for name, m, s in zip(group_names, mean_weights, std_weights):
    print(f"{name:<14} {m:>12.4f} {s:>10.4f}")

In [ ]:
# Visualise group attention weights as a bar chart
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of mean attention per group
colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0']
axes[0].bar(group_names, mean_weights, yerr=std_weights, capsize=5, color=colors)
axes[0].set_ylabel('Attention Weight')
axes[0].set_title('Mean Group Attention Weights (Test Set)')
axes[0].set_ylim(0, max(mean_weights) * 1.3)

# Distribution of attention per group (violin-style via boxplot)
bp = axes[1].boxplot(
    [all_weights[:, i].numpy() for i in range(4)],
    labels=group_names, patch_artist=True,
)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1].set_ylabel('Attention Weight')
axes[1].set_title('Group Attention Weight Distribution')

plt.tight_layout()
plt.savefig('experiments/figures_hier/group_attention_weights.png', dpi=150, bbox_inches='tight')
plt.show()

print('Saved: experiments/figures_hier/group_attention_weights.png')

## 9. Quick Results Summary

Load the experiment JSON and print a clean comparison table.

In [ ]:
import json
import glob

results_files = glob.glob("/content/thesis/experiments/results/hourly_hierarchical*.json")
results_files = [f for f in results_files if 'intermediate' not in f]

if not results_files:
    print("No results file found. Run experiments first.")
else:
    latest = max(results_files, key=os.path.getmtime)
    with open(latest) as f:
        results = json.load(f)

    model_results = results.get('model_results', results)

    print(f"{'Model':<35} {'MAPE(%)':>10} {'DirAcc(%)':>10} {'Sharpe':>10} {'DA-3cls(%)':>12} {'Sh-3cls':>10}")
    print('=' * 90)

    for name, data in model_results.items():
        agg = data['aggregated']
        print(
            f"{name:<35}"
            f" {agg['mape']['mean']:>9.2f}"
            f" {agg['directional_accuracy']['mean']:>9.2f}"
            f" {agg['sharpe_ratio']['mean']:>9.3f}"
            f" {agg['directional_accuracy_3class']['mean']:>11.2f}"
            f" {agg['sharpe_ratio_3class']['mean']:>9.3f}"
        )

    print('=' * 90)